# Q02: Quadratic Complexity in Transformers
**Topic:** LLM | **Difficulty:** Medium | **Time:** 25 min
**Sheet:** GrindLLM50

---

## Question
How does quadratic complexity affect performance of a transformer and how can we mitigate it?

## Detailed Answer

### The Problem
Self-attention computes all-pairs similarity: $\text{Attention}(Q,K,V) = \text{softmax}(QK^T/\sqrt{d_k})V$

- $QK^T$ creates an $n \times n$ attention matrix → **O(n²) time and memory**
- For $n = 128K$ context: matrix has 16 billion entries!

### Impact
- Memory: Storing attention matrix limits context length
- Compute: Quadratic FLOPs make long sequences very expensive
- Practical limit: ~512-2048 tokens without optimization

### Mitigation Strategies

**1. FlashAttention (Hardware-Aware)**
- Compute attention in **tiled blocks** without materializing full n×n matrix
- Uses kernel fusion and SRAM-aware computation
- Still exact attention, but O(n²) compute with O(n) memory
- FlashAttention-2/3 further optimized

**2. Sparse Attention Patterns**
- **Local/sliding window**: Each token only attends to nearby tokens (O(n·w))
- **Longformer**: Sliding window + global attention for special tokens
- **BigBird**: Random + local + global attention

**3. Linear Attention**
- Replace softmax(QK^T)V with φ(Q)(φ(K)^T V) — compute KV first: O(n·d²)
- Mamba/S4: State-space models — O(n·d) with recurrent formulation
- RWKV: RNN-style linear attention

**4. KV Cache Optimization**
- **Multi-Query Attention (MQA)**: Share K,V across heads → less memory
- **Grouped-Query Attention (GQA)**: Share K,V within groups of heads (LLaMA 2/3)

**5. Position Interpolation / RoPE**
- Extend context window without retraining from scratch
- NTK-aware scaling, YaRN, Dynamic NTK

### Comparison
| Approach | Complexity | Exact? | Used In |
|----------|-----------|--------|--------|
| Standard | O(n²) | Yes | Short context |
| FlashAttention | O(n²) compute, O(n) memory | Yes | Most LLMs |
| Sliding Window | O(n·w) | Approx | Mistral, Longformer |
| Linear Attention | O(n·d²) | Approx | Mamba, RWKV |
| GQA | O(n²/g) mem | Yes | LLaMA 2/3 |

## Key Takeaways
- **FlashAttention** is the most impactful — exact attention with O(n) memory
- **GQA** reduces KV cache memory by grouping heads
- **Sliding window** attention enables very long contexts (128K+)
- **Mamba/SSMs** bypass attention entirely — O(n) linear scaling